<style>
div.mermaid > svg { width: 70% !important; height: auto !important; }
</style>

# Control flow: meta loops vs staged loops

A cutlass kernel is **traced** once on the host (the Python body runs to build IR) and then
**executed** many times on the GPU. Every loop you write lands on one side of that line:

- A **meta loop** (`cutlass.range_constexpr`) is a *metaprogram* — it runs in the **Python
  interpreter, on the host, while tracing**. The body is walked `N` times right there, emitting
  `N` copies of the IR (fully unrolled). Because it is literally Python, the body can do
  *anything Python can* — compute with `int`s, branch, even `raise`.
- A **staged loop** (`cutlass.range`) traces its body **once** into a single GPU `scf.for`
  that the **hardware** iterates at run time. The loop index is a *staged* value, not a Python
  `int` — there is no concrete number to hold yet.

The cleanest way to *see* the split is a trace-time `print`: it fires once per host walk of the
body, so it counts iterations for you. This chapter needs no data or arrays — just loops and
prints.

**You'll learn:** the **trace-time vs run-time** split that governs control flow; the meta loop
**`cutlass.range_constexpr(n)`** (body runs `n` times *in Python*, fully unrolled, bound must be
a Python `int`) versus the staged loop **`cutlass.range(n)`** (body traces *once* into a GPU
`scf.for`, bound may be a dynamic `Int32`); why a meta loop can run **arbitrary Python** (we make
one `raise` mid-trace); the partial-unroll option **`cutlass.range(n, unroll=K)`**; the
compile-time branch **`if cutlass.const_expr(cond):`** versus a plain runtime `if`; and how to
read all of it straight off `print()` (host, trace-time) vs `cute.printf()` (device, run-time).

**Runs on:** any CUDA GPU.

In [ ]:
import cutlass
import cutlass.cute as cute

cutlass.cuda.initialize_cuda_context()   # once per process, before launching
print("imports OK")

## 1. Two kinds of loop

The same job — walk the indices `0 .. N-1` — written two ways. The only difference is the loop
driver, and that single word decides **where the iteration happens**.

| loop | what it is | where it iterates | bound | body trace count | emits |
|---|---|---|---|---|---|
| `cutlass.range_constexpr(N)` | **metaprogram** (Python) | **host, at trace time** | must be a Python `int` | **`N` times** | `N` straight-line copies (fully unrolled) |
| `cutlass.range(N)` | **staged** (GPU) | **device, at run time** | Python `int` *or* dynamic `Int32` | **once** | one `scf.for` the hardware loops |

We prove the split with a trace-time `print` in each body. `print` is ordinary Python, so it
fires exactly as often as the host walks the body: **`N` lines** for the meta loop, **one** for
the staged loop. In the meta loop the index `i` is a real Python `int` (so `i` accumulates a
host-side total that gets *baked* into the kernel as a constant); in the staged loop `i` is a
staged `Int32` the GPU supplies each trip, so the work happens on the device.

In [ ]:
@cute.kernel
def loop_unrolled(N: cutlass.Constexpr):
    tx, _, _ = cute.arch.thread_idx()
    if tx == 0:
        total = 0  # a plain Python int -- accumulated at TRACE time
        # N is a Constexpr (Python int), so this body runs N times right here, in Python.
        # The print fires once per iteration; `total` is computed on the host and baked in.
        for i in cutlass.range_constexpr(N):
            print(f"[trace] range_constexpr body, i = {i}  (a Python int)")
            total += i
        # `total` is now a constant -- cute.printf just emits that baked number.
        cute.printf("[device] unrolled: sum of 0..{} baked at trace = {}", N - 1, total)


@cute.jit
def loop_unrolled_host(N: cutlass.Constexpr):
    loop_unrolled(N).launch(grid=(1, 1, 1), block=(1, 1, 1))

The staged version differs in two words: `N` is a dynamic `cutlass.Int32` (the bound is decided
at *launch*, not when the kernel is written) and the loop is `cutlass.range`. Now the body
traces **once**, the accumulator is threaded through the GPU loop for you, and the sum is
computed on the **device**.

In [ ]:
@cute.kernel
def loop_staged(N: cutlass.Int32):
    tx, _, _ = cute.arch.thread_idx()
    if tx == 0:
        total = cutlass.Int32(0)
        # N is a dynamic Int32, so this is a real GPU loop: the body traces ONCE and the
        # print fires ONCE. `i` is a staged Int32 -- the GPU supplies its value each trip.
        for i in cutlass.range(N):
            print(f"[trace] range body, i is a staged {type(i).__name__} (traced once)")
            total = total + i
        cute.printf("[device] staged: sum of 0..{} computed on the GPU = {}", N - 1, total)


@cute.jit
def loop_staged_host(N: cutlass.Int32):
    loop_staged(N).launch(grid=(1, 1, 1), block=(1, 1, 1))

## 2. Run both and watch the trace

No arrays, no data setup — just stage each kernel and read the prints. The **trace** prints fire
while each kernel is *compiled* (host, Python): `N` lines from the meta loop, **one** from the
staged loop. The **device** prints fire after the launch, once we sync; both report the same sum.

In [ ]:
N = 4

print("--- staging loop_unrolled (range_constexpr) ---")
loop_unrolled_host(N)            # N trace lines fire HERE, during compile

print("\n--- staging loop_staged (range) ---")
loop_staged_host(N)              # exactly ONE trace line fires here

cutlass.cuda.stream_sync(cutlass.cuda.default_stream())   # flush the device prints

# Expected output:
# --- staging loop_unrolled (range_constexpr) ---
# [trace] range_constexpr body, i = 0  (a Python int)
# [trace] range_constexpr body, i = 1  (a Python int)
# [trace] range_constexpr body, i = 2  (a Python int)
# [trace] range_constexpr body, i = 3  (a Python int)
#
# --- staging loop_staged (range) ---
# [trace] range body, i is a staged Int32 (traced once)   <- exactly ONE line
#
# [device] unrolled: sum of 0..3 baked at trace = 6
# [device] staged: sum of 0..3 computed on the GPU = 6

## 3. The meta loop is just Python — so it can `raise`

Because `range_constexpr` runs in the Python interpreter at trace time, the body can do anything
Python can: compute, branch on Python values, call helpers — even **raise an exception**. A
raise here aborts *compilation*; it never reaches the GPU. That's the whole point of a
metaprogram — you can validate and shape the kernel with ordinary Python before a single device
instruction exists. (Try this in a `range` body instead and there is no Python `i` to test — the
check would have to become real GPU code.)

In [ ]:
@cute.kernel
def checked_unroll(N: cutlass.Constexpr):
    tx, _, _ = cute.arch.thread_idx()
    
    for i in cutlass.range_constexpr(N):
        # This runs in Python, WHILE TRACING. Ordinary Python control flow works --
        # including raising, which stops compilation before any IR is emitted for i>=3.
        if cutlass.const_expr(i == 3):
            raise ValueError(
                f"meta-loop guard tripped at i={i}: this is a Python exception raised "
                "during TRACING, before any GPU instruction is emitted"
            )
        cute.printf("[device] checked i={}", i)


@cute.jit
def checked_unroll_host(N: cutlass.Constexpr):
    checked_unroll(N).launch(grid=(1, 1, 1), block=(1, 1, 1))

In [ ]:
print("N=3: the meta loop never reaches i==3, so it traces + runs cleanly:")
checked_unroll_host(3)
cutlass.cuda.stream_sync(cutlass.cuda.default_stream())

print("\nN=8: the meta loop hits i==3 and raises -- DURING tracing, on the host:")
try:
    checked_unroll_host(8)   # raises while the Python body is being walked
except ValueError as e:
    print(f"  caught at trace time: {e}")

# Expected output:
# N=3: ... traces + runs cleanly:
# [device] checked i=0
# [device] checked i=1
# [device] checked i=2
#
# N=8: ... raises -- DURING tracing, on the host:
#   caught at trace time: meta-loop guard tripped at i=3: this is a Python exception ...

## 4. Partial unroll: `range(N, unroll=K)`

A staged loop can still be unrolled — just not all the way. `cutlass.range(N, unroll=K)` keeps a
real GPU loop (the bound stays dynamic, the body still traces **once**) but asks the compiler to
emit `K` copies of the body per trip, so the loop runs `N // K` times with `K`-wide bodies plus a
short remainder for the last `N % K` iterations. It is the middle ground: fewer loop-control
instructions and more instruction-level parallelism than a rolled loop, without the code-size
blow-up of a full unroll. The result is identical — only the generated loop shape changes.
(`unroll=1` disables unrolling; `unroll=0`, like leaving `unroll` out, lets the compiler pick the
factor; `unroll_full=True` only flattens a loop whose trip count is known at compile time.)

In [ ]:
@cute.kernel
def loop_unroll4(N: cutlass.Int32):
    tx, _, _ = cute.arch.thread_idx()
    if tx == 0:
        total = cutlass.Int32(0)
        # Still a GPU loop (N is dynamic) -- the body traces once; the COMPILER emits 4
        # bodies per trip. So the trace print still fires exactly once.
        for i in cutlass.range(N, unroll=4):
            print(f"[trace] unroll=4 body, traced once (i is staged {type(i).__name__})")
            total = total + i
        cute.printf("[device] unroll=4: sum of 0..{} = {}", N - 1, total)


@cute.jit
def loop_unroll4_host(N: cutlass.Int32):
    loop_unroll4(N).launch(grid=(1, 1, 1), block=(1, 1, 1))


loop_unroll4_host(N)
cutlass.cuda.stream_sync(cutlass.cuda.default_stream())

# Expected output:
# [trace] unroll=4 body, traced once (i is staged Int32)   <- still ONE trace line
# [device] unroll=4: sum of 0..3 = 6

## 5. Two kinds of branch

Branches split the same way as loops:

- `if cutlass.const_expr(cond):` — `cond` is a Python value fixed at trace time, so the host
  picks **one** side and compiles only that; the other is never traced. A trace-time `print` in
  each side proves it — only the taken side fires.
- a plain `if <staged>:` — the condition is a per-thread device value, so **both** sides compile
  and the GPU chooses per thread (a real `scf.if`). You still write a plain Python `if`; the DSL
  stages it because the condition is dynamic.

No arrays here either — we branch on a `Constexpr` flag and on the per-thread `tidx`.

In [ ]:
@cute.kernel
def branch_demo(USE_DOUBLE: cutlass.Constexpr):
    tx, _, _ = cute.arch.thread_idx()
    # COMPILE-TIME branch: USE_DOUBLE is a Python bool, so exactly ONE side is traced and
    # emitted -- the other side's print never fires. `factor` is baked from the taken side.
    if cutlass.const_expr(USE_DOUBLE):
        print("[trace] const_expr branch: doubling side compiled in")
        factor = 2
    else:
        print("[trace] const_expr branch: identity side compiled in")
        factor = 1
    # RUNTIME branch: tx is a staged value, so BOTH sides compile and the GPU picks per
    # thread. Plain Python if -- the DSL stages it into a real scf.if.
    if tx == 0:
        cute.printf("[device] thread {} -> first lane (baked factor = {})", tx, factor)
    else:
        cute.printf("[device] thread {} -> other lane (baked factor = {})", tx, factor)


@cute.jit
def branch_demo_host(USE_DOUBLE: cutlass.Constexpr):
    branch_demo(USE_DOUBLE).launch(grid=(1, 1, 1), block=(2, 1, 1))

In [ ]:
for use_double in (True, False):
    print(f"--- staging branch_demo(USE_DOUBLE={use_double}) ---")
    branch_demo_host(use_double)        # only the TAKEN const_expr side prints at trace
    cutlass.cuda.stream_sync(cutlass.cuda.default_stream())
    print()

# Expected output (only ONE const_expr print per staging -- the taken side):
# --- staging branch_demo(USE_DOUBLE=True) ---
# [trace] const_expr branch: doubling side compiled in
# [device] thread 0 -> first lane (baked factor = 2)
# [device] thread 1 -> other lane (baked factor = 2)
#
# --- staging branch_demo(USE_DOUBLE=False) ---
# [trace] const_expr branch: identity side compiled in
# [device] thread 0 -> first lane (baked factor = 1)
# [device] thread 1 -> other lane (baked factor = 1)

## Try it yourself

1. **Count the trace prints.** Bump `N` to 8 and re-run section 2: the meta loop now prints 8
   trace lines, the staged loop still prints exactly one.
2. **Break the constexpr bound.** Change `loop_unrolled`'s `N` from `cutlass.Constexpr` to
   `cutlass.Int32` and pass a runtime value — `range_constexpr` raises, because its bound must
   be a Python `int` known at trace time. Switching that loop to `cutlass.range(N)` fixes it.
3. **Move the guard into a staged loop.** Put the `if i == 3: raise ...` from section 3 inside a
   `cutlass.range(N)` loop. It does *not* raise per-trip — `i` is a staged value, so `i == 3` is
   a device comparison, not a Python one; the meta-only trick is gone.
4. **Make the const_expr branch runtime.** In section 5, drop `cutlass.const_expr(...)` and pass
   `USE_DOUBLE` as a `cutlass.Int32` flag compared `> 0` — now *both* sides compile (both trace
   prints fire) and the GPU chooses at run time, just like the `tx == 0` guard already does.